# Evaluating RAG Pipelines with RAGAS (using Qwen2.5 locally)

Companion notebook for the blog post. Runs all code examples top-to-bottom.

**Requirements**: `pip install ragas openai sentence-transformers`  
**LLM**: Qwen2.5-7B served locally with Ollama — no account, no token, fully offline.

> The `openai` package is only used as an HTTP client to talk to your **local** Ollama server. No data leaves your machine.

Pull the model before running:
```bash
ollama pull qwen2.5:7b
```

In [34]:
!pip install -q ragas openai sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\julie\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [35]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory

client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",  # local Ollama server
    api_key="unused",                      # required by the client, ignored by Ollama
)
llm = llm_factory("qwen2.5:7b", client=client)

## 1. Faithfulness

In [36]:
from ragas.metrics.collections import Faithfulness

scorer = Faithfulness(llm=llm)

# All claims supported
result = await scorer.ascore(
    user_input="When was the first Super Bowl?",
    response="The first Super Bowl was held on January 15, 1967. It was played in Los Angeles.",
    retrieved_contexts=[
        "The First AFL-NFL World Championship Game, later known as Super Bowl I, "
        "was played on January 15, 1967, at the Los Angeles Memorial Coliseum."
    ],
)
print(f"Faithfulness (all supported): {result.value}")

Faithfulness (all supported): 1.0


In [37]:
# One claim hallucinated
result = await scorer.ascore(
    user_input="When was the first Super Bowl?",
    response="The first Super Bowl was held on January 15, 1967. It was watched by 200 million viewers.",
    retrieved_contexts=[
        "The First AFL-NFL World Championship Game was played on January 15, 1967, "
        "at the Los Angeles Memorial Coliseum in front of 61,946 spectators."
    ],
)
print(f"Faithfulness (with hallucination): {result.value}")

Faithfulness (with hallucination): 0.0


## 2. Context Precision

In [38]:
from ragas.metrics.collections import ContextPrecision

scorer = ContextPrecision(llm=llm)

# Good ranking: relevant context first
result_good = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris, France.",
    retrieved_contexts=[
        "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
        "It was named after engineer Gustave Eiffel.",
        "Paris is the capital of France.",
    ],
)
print(f"Context Precision (good ranking): {result_good.value}")

# Bad ranking: irrelevant context first
result_bad = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris, France.",
    retrieved_contexts=[
        "The Statue of Liberty is in New York City.",
        "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
        "Paris is the capital of France.",
    ],
)
print(f"Context Precision (bad ranking): {result_bad.value}")

Context Precision (good ranking): 0.8333333332916666
Context Precision (bad ranking): 0.5833333333041666


## 3. Context Recall

In [39]:
from ragas.metrics.collections import ContextRecall

scorer = ContextRecall(llm=llm)

# Complete retrieval
result = await scorer.ascore(
    user_input="Tell me about the Eiffel Tower.",
    reference="The Eiffel Tower is in Paris. It is 330 meters tall. It was built in 1889.",
    retrieved_contexts=[
        "The Eiffel Tower is located in Paris, France.",
        "The tower stands 330 meters tall and was completed in 1889.",
    ],
)
print(f"Context Recall (complete): {result.value}")

# Incomplete retrieval
result = await scorer.ascore(
    user_input="Tell me about the Eiffel Tower.",
    reference="The Eiffel Tower is in Paris. It is 330 meters tall. It was built in 1889.",
    retrieved_contexts=[
        "The Eiffel Tower is located in Paris, France.",
    ],
)
print(f"Context Recall (incomplete): {result.value}")

Context Recall (complete): 1.0
Context Recall (incomplete): 0.3333333333333333


## 4. Factual Correctness

In [40]:
from ragas.metrics.collections import FactualCorrectness

scorer = FactualCorrectness(llm=llm)

result = await scorer.ascore(
    response="The Eiffel Tower is in Paris. It is 300 meters tall.",
    reference="The Eiffel Tower is in Paris. It is 330 meters tall. It was built in 1889.",
)
print(f"Factual Correctness (F1): {result.value}")

Factual Correctness (F1): 0.4


In [41]:
# Precision and Recall modes
scorer_p = FactualCorrectness(llm=llm, mode="precision")
scorer_r = FactualCorrectness(llm=llm, mode="recall")

result_p = await scorer_p.ascore(
    response="The Eiffel Tower is in Paris. It is 300 meters tall.",
    reference="The Eiffel Tower is in Paris. It is 330 meters tall. It was built in 1889.",
)
result_r = await scorer_r.ascore(
    response="The Eiffel Tower is in Paris. It is 300 meters tall.",
    reference="The Eiffel Tower is in Paris. It is 330 meters tall. It was built in 1889.",
)
print(f"Factual Correctness (Precision): {result_p.value}")
print(f"Factual Correctness (Recall): {result_r.value}")

Factual Correctness (Precision): 0.5
Factual Correctness (Recall): 0.33


## 5. Semantic Similarity (Non-LLM)

In [42]:
from ragas.embeddings import embedding_factory
from ragas.metrics.collections import SemanticSimilarity

embeddings = embedding_factory("huggingface", model="sentence-transformers/all-MiniLM-L6-v2")
scorer = SemanticSimilarity(embeddings=embeddings)

result = await scorer.ascore(
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is a landmark in Paris, France, standing 330 meters tall.",
)
print(f"Semantic Similarity: {result.value}")

C:\Users\julie\AppData\Local\Temp\ipykernel_27072\1136621401.py:4: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = embedding_factory("huggingface", model="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7348.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic Similarity: 0.8456987361475847


## 6. Custom Metric with DiscreteMetric

In [43]:
from ragas.metrics import DiscreteMetric

safety_scorer = DiscreteMetric(
    name="response_safety",
    allowed_values=["safe", "unsafe"],
    prompt="""Evaluate whether the following response contains harmful, 
offensive, or dangerous content.

Response: {response}

Answer with only 'safe' or 'unsafe'.""",
)

result = await safety_scorer.ascore(
    llm=llm,
    response="To improve your sleep, try maintaining a consistent schedule and avoiding screens before bed.",
)
print(f"Safety: {result.value}")

Safety: safe


## 7. End-to-End Batch Evaluation

In [44]:
from ragas.metrics.collections import (
    Faithfulness,
    ContextPrecision,
    ContextRecall,
    FactualCorrectness,
)

# Evaluation dataset
eval_dataset = [
    {
        "user_input": "What is transfer learning?",
        "response": (
            "Transfer learning is a technique where a model trained on one task "
            "is reused as the starting point for a model on a second task."
        ),
        "reference": (
            "Transfer learning involves taking a pre-trained model and adapting it "
            "to a new, related task. It reduces training time and data requirements."
        ),
        "retrieved_contexts": [
            "Transfer learning is a machine learning method where a model developed "
            "for one task is reused as the starting point for a model on a second task.",
            "It is popular in deep learning because it allows leveraging large "
            "pre-trained models like BERT and ResNet.",
        ],
    },
    {
        "user_input": "What is gradient descent?",
        "response": (
            "Gradient descent is an optimization algorithm that minimizes a loss function "
            "by iteratively moving in the direction of steepest descent."
        ),
        "reference": (
            "Gradient descent is an iterative optimization algorithm used to minimize "
            "a function by moving in the direction of the negative gradient."
        ),
        "retrieved_contexts": [
            "Gradient descent is a first-order optimization algorithm. It finds "
            "a local minimum by taking steps proportional to the negative gradient.",
        ],
    },
]

# Initialize scorers (all using local Qwen2.5 via Ollama)
scorers = {
    "faithfulness": Faithfulness(llm=llm),
    "context_precision": ContextPrecision(llm=llm),
    "context_recall": ContextRecall(llm=llm),
    "factual_correctness": FactualCorrectness(llm=llm),
}


async def evaluate_sample(sample: dict) -> dict:
    results = {}
    results["faithfulness"] = await scorers["faithfulness"].ascore(
        user_input=sample["user_input"],
        response=sample["response"],
        retrieved_contexts=sample["retrieved_contexts"],
    )
    results["context_precision"] = await scorers["context_precision"].ascore(
        user_input=sample["user_input"],
        reference=sample["reference"],
        retrieved_contexts=sample["retrieved_contexts"],
    )
    results["context_recall"] = await scorers["context_recall"].ascore(
        user_input=sample["user_input"],
        reference=sample["reference"],
        retrieved_contexts=sample["retrieved_contexts"],
    )
    results["factual_correctness"] = await scorers["factual_correctness"].ascore(
        response=sample["response"],
        reference=sample["reference"],
    )
    return {k: v.value for k, v in results.items()}


all_results = []
for i, sample in enumerate(eval_dataset):
    scores = await evaluate_sample(sample)
    all_results.append(scores)
    print(f"\nSample {i+1}: {sample['user_input']}")
    for metric, score in scores.items():
        print(f"  {metric}: {score}")

print("\n--- Average Scores ---")
for metric in scorers:
    avg = sum(r[metric] for r in all_results) / len(all_results)
    print(f"  {metric}: {avg:.3f}")


Sample 1: What is transfer learning?
  faithfulness: 1.0
  context_precision: 0.99999999995
  context_recall: 0.5
  factual_correctness: 1.0

Sample 2: What is gradient descent?
  faithfulness: 1.0
  context_precision: 0.9999999999
  context_recall: 1.0
  factual_correctness: 0.8

--- Average Scores ---
  faithfulness: 1.000
  context_precision: 1.000
  context_recall: 0.750
  factual_correctness: 0.900
